In [0]:
import pandas as pd

In [0]:
df = pd.read_csv("../data/marvel_characters_dataset.csv")
print(df.shape)
df.head()

In [0]:
df.info()

In [0]:
df.isnull().sum().sort_values(ascending=False)

In [0]:
# check target distribution
df["Alive"].value_counts(normalize=True)

### Data Cleaning

In [0]:
# clean the target column
def clean_target(val):
    if pd.isna(val):
        return None
    val = str(val).lower()
    if 'dead' in val:
        return 0
    elif 'alive' in val or 'ghost' in val or 'unknown' in val:
        return 1
    return 1
df['Alive_clean'] = df['Alive'].apply(clean_target)
df['Alive_clean'].value_counts(normalize=True)

In [0]:
# Drop rows with missing target
# alive column initially showed 1 missing value in original data
df_clean = df.dropna(subset=['Alive_clean'])
num_features = ['Height (m)', 'Weight (kg)']
cat_features = ['Gender', 'Identity', 'Marital Status', 'Origin']
target = 'Alive_clean'
x = df_clean[cat_features + num_features]
y = df_clean[target]

print(f"Feature shape: {x.shape}")
print(f"Target shape: {y.shape}")
print(f"missing values in features:\n{x.isnull().sum()}")

In [0]:
# train-test-split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train target distribution: \n{y_train.value_counts(normalize=True)}")

In [0]:
# First, let's see what we need to do:
print("Numerical features need:")
print("  1. Impute missing values (median)")
print("  2. Scale to similar range (optional for tree models)")

print("\nCategorical features need:")
print("  1. Impute missing values ('Unknown')")
print("  2. Encode to numbers (OneHot or Ordinal)")

print("\nColumnTransformer lets us do BOTH in one step!")

In [0]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import pandas as pd

class CategoryConverter(BaseEstimator, TransformerMixin):
    "converts columns to pandas category codes for lightgbm"
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X = pd.DataFrame(X).copy()
        for col in X.columns:
            X[col]= X[col].astype('category').cat.codes
        return X

# define transformation for each column type
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('to_codes', CategoryConverter())
])
# combine all with column transformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, num_features),
        ('cat', categorical_transformer, cat_features)
    ])
print('preprocessor created')
print(f"transforming {len(num_features)} numeric + {len(cat_features)} categorical features")

# fitting preprocessor on the data
X_train_transformed = preprocessor.fit_transform(X_train)
print(f"Original shape: {X_train.shape}")
print(f"Transformed shape: {X_train_transformed.shape}")

In [0]:
pip install lightgbm

In [0]:
from lightgbm import LGBMClassifier

full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LGBMClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=6,
        random_state=42,
        verbose=-1
    ))
])
print("full pipeline created")
print(full_pipeline)

In [0]:
from sklearn.metrics import accuracy_score, classification_report
# go train
full_pipeline.fit(X_train, y_train)
# now predict
y_pred = full_pipeline.predict(X_test)
# then evaluate
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Dead', 'Alive']))

In [0]:
import mlflow
from mlflow.models import infer_signature

# connect  to databricks mlflow
mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks-uc")

# create the experiment
experiment = mlflow.set_experiment(experiment_name="/Users/enochobey@outlook.com/Marvel_Pipeline_Tutorial")

# Re-train for a fresh run (to ensure consistent state)
full_pipeline.fit(X_train, y_train)
y_pred = full_pipeline.predict(X_test)

signature = infer_signature(
    model_input=X_train,
    model_output=full_pipeline.predict(X_train)
)

print("Signature captures:")
print(f"  Inputs: {signature.inputs}")
print(f"  Outputs: {signature.outputs}")

In [0]:
with mlflow.start_run(run_name="marvel_lgbm_pipeline_v2") as run:
    # logging with automatic evaluation
    mlflow.log_params({
        "n_estimators": 100,
        "learning_rate": 0.1,
        "max_epth": 6,
        "num_features": str(num_features),
        "cat_features": str(cat_features)
    })

    # log the model with signatures
    model_info = mlflow.sklearn.log_model(
        sk_model=full_pipeline,
        artifact_path="marvel-pipeline-model",
        signature=signature,
        input_example=X_test.iloc[0:5]
    )

    # Auto log all metrics
    eval_data = X_test.copy()
    eval_data["Alive_clean"] = y_test # added the target to the eval data
    result = mlflow.models.evaluate(
        model=model_info.model_uri,
        data=eval_data,
        targets="Alive_clean",
        model_type="classifier",
        evaluators=["default"]
    )

    print(f"Run ID: {run.info.run_id}")
    print(f"Model URI: {model_info.model_uri}")
    print(f"\nAuto-computed metrics:")
    for metric, value in result.metrics.items():
        print(f" {metric}: {value:.4f}")

In [0]:
# load the model back from mlflow
loaded_model = mlflow.sklearn.load_model(model_info.model_uri)
# predict with loaded model
test_predictions = loaded_model.predict(X_test.iloc[0:5])
print(f"Predictions: {test_predictions}")
print(f"Actual:      {y_test.iloc[0:5].values}")

## Exercise 4: MlflowClient + Model Registry

In [0]:
model_info.model_uri

In [0]:
# register the model
model_name = "mlops_dev.marvel_characters.marvel_survival_model" #catalog.schema.model_name
registered_model = mlflow.register_model(
    model_info.model_uri, # from the logged model
    name=model_name,
    tags={"author":"enoch", "project":"marvel-tutorial"}
)
print(f"model registered")
print(f"name: {registered_model.name}")
print(f"   Version: {registered_model.version}")

### MlflowClient is used AFTER registration for:

* Setting aliases: client.set_registered_model_alias()
* Getting versions: client.get_model_version_by_alias()
* Updating descriptions: client.update_model_version()
* Transitioning stages (legacy): client.transition_model_version_stage()

In [0]:
from mlflow import MlflowClient

# MlflowClient gives you programmatic access to MLflow
client = MlflowClient()

# Why use MlflowClient instead of mlflow.xxx?
# - More control over model registry operations
# - Set aliases, descriptions, tags on versions
# - Query model versions programmatically
# - Production deployment workflows

print("MlflowClient initialized!")
print("Available methods for model registry:")
print("  - client.get_registered_model(name)")
print("  - client.get_model_version(name, version)")
print("  - client.set_registered_model_alias(name, alias, version)")
print("  - client.get_model_version_by_alias(name, alias)")

In [0]:
# Aliases let you reference models by purpose, not version number
# Common aliases: "champion", "challenger", "latest-model", "staging", "production"
client.set_registered_model_alias(
    name=model_name,
    alias="latest_model",
    version=registered_model.version
)
print(f"Alias 'latest-model' set to version {registered_model.version}")

In [0]:
# In production, you load by ALIAS, not version number
# allows you to update the model without changing code
model_by_alias = mlflow.sklearn.load_model(f"models:/{model_name}@latest_model")
# test it
sample_predictions = model_by_alias.predict(X_test.iloc[0:3])
print(f"Predictions from 'latest-model': {sample_predictions}")

In [0]:
# Get model info using client
model_version = client.get_model_version_by_alias(
    name=model_name,
    alias="latest_model"
)

print(f"Model Version Details:")
print(f"  Version: {model_version.version}")
print(f"  Status: {model_version.status}")
print(f"  Run ID: {model_version.run_id}")
print(f"  Model ID: {model_version.model_id}")

## Exercise 5: Custom PyFunc Wrapper
**Why wrap a model?**  
*Transform outputs (numbers → human-readable labels)*  
*Add pre/post-processing logic*  
*Bundle multiple models together*  
*Custom business logic in predictions*

In [0]:
# current model output:
raw_predictions = full_pipeline.predict(X_test.iloc[0:5])
print(f"Raw output: {raw_predictions}")
print(f"Type: {type(raw_predictions)}")

In [0]:
import mlflow.pyfunc
import pandas as pd
import numpy as np

class MarvelSurvivalWrapper(mlflow.pyfunc.PythonModel):
    """wrapper return human readable predictions"""
    def load_context(self, context):
        """called when the model is loaded. load the underlying model."""
        # context.artifacts contains paths to saved artifacts
        self.model = mlflow.sklearn.load_model(context.artifacts["pipeline"])

    def predict(self, context, model_input):
        """transform predictions to human readable format"""
        # get the raw predictions (0 or 1)
        raw_preds = self.model.predict(model_input)
        # convert to human readable format
        labels = ["dead" if p == 0 else "alive" for p in raw_preds]

        return {"survival prediction": labels}
    
print("Wrapper class defined!")
print("Methods:")
print("  - load_context(): loads the underlying model")
print("  - predict(): returns custom formatted output")

In [0]:
wrapped_model_uri = f"models:/{model_name}@latest_model"
with mlflow.start_run(run_name="marvel_pyfunc_wrapper") as run:
    # log pyfunc model with reference to underlying model
    wrapper_info = mlflow.pyfunc.log_model(
        artifact_path="wrapped-model",
        python_model=MarvelSurvivalWrapper(), # custom class
        artifacts={"pipeline":wrapped_model_uri},
        signature=infer_signature(
            model_input=X_test.iloc[0:5],
            model_output={"Survival prediction": ["alive", "dead", "alive", "alive", "dead"]}
        ),
        input_example=X_test.iloc[0:3]
    )
    print(f"Wrapped model logged!")
    print(f"Model URI: {wrapper_info.model_uri}")

In [0]:
# load the pyfunc model
loaded_pyfunc = mlflow.pyfunc.load_model(wrapper_info.model_uri)
# predict - now returns human-readable output
result = loaded_pyfunc.predict(X_test.iloc[0:5])
print(f"Wrapped model output:\n{result}")

# compare with raw model
print(f"\nraw model output: {raw_predictions}")

In [0]:
# access the underlying model - optional
unwrapped = loaded_pyfunc.unwrap_python_model()
print(f"unwrapped model type: {type(unwrapped)}")

# call the unwrapped model directly
direct_result = unwrapped.predict(context=None, model_input=X_test.iloc[:3])
print(f"direct call result: {direct_result}")